# FlyLIF模块化任务清单

## 背景
- 项目：Drosophila Brain LIF Model复现
- 当前状态：Jupyter notebook原型（2000+行，混合中英文）
- 网络规模：139K neurons, 2.7M synapses
- 性能：网络构建8秒，并行验证8.39×加速（3核）
- 目标：模块化代码 → GitHub发布 → 48核云端部署

---

## 任务1: 提取参数模块

**文件**: `flylif/core/parameters.py`

**提取内容**: notebook Section 2的`DEFAULT_PARAMS`字典

**要求**:
- 转为独立模块
- 保持Brian2单位（mV, ms, Hz）
- 添加docstring说明每个参数

**验证**: 在notebook中`from flylif.core.parameters import DEFAULT_PARAMS`成功

---

## 任务2: 提取网络构建模块

**文件**: `flylif/core/network.py`

**提取内容**: 
- `determine_neuron_types_fast()`
- `build_network()` 
- 所有Section 5的辅助函数

**关键点**:
- 移除全局变量依赖（DATA, CONFIG等改为参数传递）
- 保持函数签名不变
- 包含所有依赖的子函数

**验证**: 
```python
from flylif.core.network import build_network
NET = build_network(data=DATA, pre_col=PRE_COL, ...)  # 应与原notebook结果一致
```

---

## 任务3: 提取仿真模块

**文件**: `flylif/core/simulation.py`

**提取内容**:
- `run_simulation()`
- `run_silencing_batch()`
- Section 5.3-5.4的函数

**要求**:
- 处理Brian2对象导入
- 保持API兼容

**验证**: 运行单个仿真，对比spike数

---

## 任务4: 提取数据加载模块

**文件**: `flylif/core/data_loader.py`

**提取内容**:
- `load_connectivity_data()`
- `load_all_data()`
- Section 4的函数

**验证**: 加载数据并检查完整性

---

## 任务5: 创建并行测试脚本

**文件**: `scripts/test_parallel_exp1.py`

**内容**:
- 导入上述模块
- 实现worker函数（每worker重建网络）
- 使用joblib.Parallel
- 参数：21神经元 × 5频率 × 2 trials

**预期**: 6核加速比3-5×，耗时2-3分钟

---

## 任务6: 创建实验1完整脚本

**文件**: `scripts/run_exp1_full.py`

**参数**: 21神经元 × 19频率 × 10 trials（目标参数）

**功能**:
- 命令行参数支持（频率范围、trials数）
- 进度显示
- 结果保存（.pkl + .csv）

---

## 任务7: Checkpoint机制

**文件**: `flylif/parallel/checkpoint.py`

**功能**:
- 保存已完成任务ID
- 断点续传
- 进度查询

---

## 使用说明

**提问格式**（每次只做一个任务）:
```
请帮我完成任务N: [任务名称]

附加信息:
- notebook中相关代码在Section X
- [如有特殊要求]
```

**Claude将提供**:
1. 完整代码文件
2. 测试代码
3. 验证步骤

**优先级**:
- 任务1-3: 必须（核心功能）
- 任务4: 必须（数据依赖）
- 任务5: 必须（验证并行）
- 任务6-7: 可选（锦上添花）

**预估时间**:
- 任务1: 20分钟
- 任务2: 2小时
- 任务3: 1小时
- 任务4: 30分钟
- 任务5: 30分钟
- 任务6: 1小时
- 任务7: 1小时

**总计**: 约6小时（可分2天完成）

# 你的当前位置应该是：
/Users/charlottel/MyLibrary/connectome/LIFmodel/LIF_simulation/

# 创建结构：
LIF_simulation/
├── flylif/                          ← 在这里创建
│   ├── __init__.py
│   ├── core/
│   │   ├── __init__.py
│   │   ├── parameters.py
│   │   ├── network.py
│   │   ├── simulation.py
│   │   └── data_loader.py
│   └── parallel/
│       └── __init__.py
├── scripts/                         ← 同级
│   └── test_parallel_exp1.py
├── repo_flylif_model-Copy1.ipynb    ← 你的notebook
├── data_783/                        ← 已有
└── lif_simulation/                  ← 已有

# FlyLIF优化版模块化任务清单

## 背景更新
- **关键发现**: 并行测试显示数据传输是瓶颈（加速比仅2.58×）
- **优化策略**: 预处理neuron_sign，删除6个神经递质列，数据缩减68% (1.1GB→0.35GB)
- **预期效果**: 加速比提升至3.5-4.0×

---

## 任务1: 参数模块（无变化）

**文件**: `flylif/core/parameters.py`

**提取内容**: notebook Section 2的`DEFAULT_PARAMS`

**要求**: 
- 保持Brian2单位
- 添加docstring

**代码量**: 50行

---

## 任务2: 数据加载模块（⭐核心优化）

**文件**: `flylif/core/data_loader.py`

**新增功能**:

1. **基础加载**（已有）
   - `load_connectivity_data()` - chunk streaming

2. **预处理函数**（新增）
   ```python
   def compute_neuron_signs(df_conn, nt_prob_cols):
       """计算兴奋性(+1)/抑制性(-1)
       基于 GABA + Glutamate 概率 > 0.5
       返回添加了'neuron_sign'列的df_conn
       """
   ```

3. **主加载函数**（增强）
   ```python
   def load_all_data(config, optimize_for='simulation'):
       """
       optimize_for: 
       - 'simulation': 预处理+删除神经递质列 (0.35GB)
       - 'analysis': 预处理但保留所有列 (1.1GB)  
       - 'raw': 不处理 (1.1GB)
       """
   ```

**数据结构变化**:
```python
# optimize_for='simulation'时
df_conn.columns = ['pre_pt_root_id', 'post_pt_root_id', 'syn_count', 'neuron_sign']

# optimize_for='analysis'时
df_conn.columns = [...原有10列..., 'neuron_sign']
```

**代码量**: 120-150行

**测试**: 
```python
DATA = load_all_data(config)
assert 'neuron_sign' in DATA['df_conn'].columns
assert len(DATA['df_conn'].columns) == 4  # 仅4列
```

---

## 任务3: 网络构建模块（需适配）

**文件**: `flylif/core/network.py`

**修改点**:

1. **`determine_neuron_types_fast()`适配**
   ```python
   # 新增：检查预处理
   if 'neuron_sign' in df_conn.columns:
       # 使用预处理的sign（快速路径）
       signs = df_conn_filtered['neuron_sign'].values
   else:
       # 原逻辑（向后兼容）
       # 基于GABA/Glut概率计算
   ```

2. **`build_network()`函数签名保持不变**
   - 但内部逻辑简化（利用neuron_sign列）
   - 减少对nt_prob_cols的依赖

**代码量**: 300-400行（主要是复制现有代码）

**测试**: 
```python
# 使用优化数据
NET = build_network(data=DATA_optimized, ...)
# 使用未优化数据（兼容性）
NET = build_network(data=DATA_raw, ...)
# 结果应一致
```

---

## 任务4: 仿真模块（无需改动）

**文件**: `flylif/core/simulation.py`

**提取内容**: 
- `run_simulation()`
- `run_silencing_batch()`

**要求**: 直接复制，无需修改（不依赖数据格式）

**代码量**: 200行

---

## 任务5: 并行测试脚本（更新）

**文件**: `scripts/test_parallel_exp1.py` 或 notebook cells

**关键改动**:
```python
# 使用优化数据
data = load_all_data(CONFIG, optimize_for='simulation')  # 0.35GB

# Worker函数传递更小的data
def run_frequency_worker(freq, neu_exc, data, ...):
    net = build_network(data=data, ...)  # data已优化
```

**预期**: 加速比从2.58×提升至3.5-4.0×

---

## 验证流程

### Step 1: 测试数据优化效果（在notebook）

```python
# 新cell
DATA_old = DATA.copy()  # 备份
DATA = load_all_data(CONFIG, optimize_for='simulation')

print(f"优化前: {DATA_old['df_conn'].memory_usage(deep=True).sum()/1e6:.0f}MB")
print(f"优化后: {DATA['df_conn'].memory_usage(deep=True).sum()/1e6:.0f}MB")

# 重新运行并行测试
# 对比加速比变化
```

### Step 2: 验证科学正确性

```python
# 使用优化数据重建网络
NET_optimized = build_network(data=DATA, ...)

# 运行相同实验
result_optimized = run_simulation(NET_optimized, neu_exc=NEU_SUGAR, ...)

# 对比原始结果
assert result_optimized['n_active'] == result_baseline['n_active']
# 允许微小差异（随机性）
```

---

## 提问模板（其他chat使用）

### 任务2示例

```
任务2: 创建data_loader.py（优化版）

要求：
1. 包含compute_neuron_signs()预处理函数
2. load_all_data()支持3种模式（simulation/analysis/raw）
3. simulation模式删除6个神经递质列，缩减至4列

参考代码：notebook Section 4
- load_connectivity_data()函数
- load_all_data()函数
- NT_PROB_COLS识别逻辑

预期输出：
- 完整的data_loader.py文件
- docstring说明每个模式的数据大小
- 测试代码
```

### 任务3示例

```
任务3: 创建network.py（适配预处理数据）

要求：
1. determine_neuron_types_fast()检查neuron_sign列
2. 如果存在neuron_sign→直接使用（快速路径）
3. 如果不存在→原逻辑计算（兼容旧数据）

参考代码：notebook Section 5
- build_network()完整函数
- determine_neuron_types_fast()
- 所有辅助函数

注意：
- 保持函数签名不变
- 添加verbose参数控制打印
```

---

## 优先级（更新）

```
必做：
├─ 任务1: parameters.py (20分钟)
├─ 任务2: data_loader.py (1小时) ← 包含优化逻辑
├─ 任务3: network.py (1.5小时) ← 适配预处理数据
└─ 任务4: simulation.py (1小时)

验证：
└─ 任务5: 并行测试 (10分钟运行+分析)

预期总时间：4-5小时
预期加速比：3.5-4.0×（vs 当前2.58×）
```

---

## 关键差异（vs 原计划）

| 项目 | 原计划 | 优化版 | 原因 |
|------|--------|--------|------|
| data_loader.py | 50行，30分钟 | 150行，1小时 | +预处理逻辑 |
| 数据传输量 | 1.1GB | 0.35GB | 预计算sign |
| 并行加速比 | 2-3× | 3.5-4× | 减少传输瓶颈 |
| 向后兼容 | - | 支持3种模式 | 灵活性 |

---

## 给其他chat的完整prompt示例

**复制以下内容发送：**

```
创建优化版flylif模块（4个文件）

背景：
- Drosophila brain LIF model，139K neurons
- 并行测试显示数据传输瓶颈（1.1GB导致加速比仅2.58×）
- 需要预处理neuron_sign，缩减至0.35GB

任务2: data_loader.py（优先）
├─ compute_neuron_signs(): 基于GABA+Glut>0.5判断兴奋/抑制
├─ load_all_data(optimize_for='simulation'/'analysis'/'raw')
└─ simulation模式：删除6个神经递质列，只保留4列

任务3: network.py（适配预处理）
├─ determine_neuron_types_fast(): 检查neuron_sign列
├─ 存在→直接用，不存在→原逻辑计算
└─ build_network(): 支持两种数据格式

任务1: parameters.py（简单）
└─ 提取DEFAULT_PARAMS字典

任务4: simulation.py（不变）
└─ 提取run_simulation()等函数

参考：我有完整notebook代码，会提供Section 2/4/5相关代码片段

请先生成任务2（data_loader.py），包含完整代码+测试+docstring
```